# Анализ результатов — глава 4 диплома

Считаем все таблицы главы 4 на готовых per-example метриках из
`results/*.jsonl`. Каждая запись уже содержит поля `stage3`, `stage4`,
`stage5`, посчитанные `pipeline.eval` во время прогона. Этот ноутбук
только агрегирует, делает срезы и экспортирует таблицы в LaTeX.

Все вычисления проходят через `pipeline.eval.aggregate_stage{3,4,5}`,
а агрегация по группам — через хелперы из `_chapter4_helpers`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from collections import defaultdict

import sys
sys.path.insert(0, str(Path.cwd()))

from _chapter4_helpers import (
    REPO,
    load_run,
    join_gold,
    agg_stages,
    agg_by,
    fp_category_counts,
    fmt,
    fmt_pct,
    tex_table,
    write_tex,
    predictions_total,
)

DATA = REPO / "data"
RESULTS = REPO / "results"
TABLES_OUT = REPO / "notebooks" / "tables"
TABLES_OUT.mkdir(parents=True, exist_ok=True)

DEV_PATH = DATA / "dev_dataset.jsonl"
TEST_PATH = DATA / "test_dataset.jsonl"

## 1. Реестр прогонов

In [ ]:
RUNS = {
    "baseline_ds":  RESULTS / "dev_baseline_or_deepseek.jsonl",
    "baseline_gem": RESULTS / "dev_baseline_or_gemini.jsonl",
    "cot_ds":       RESULTS / "cot_or_deepseek.jsonl",
    "cot_gem":      RESULTS / "cot_or_gemini.jsonl",
    "fewshot_ds":   RESULTS / "dev_fewshot_or_deepseek.jsonl",
    "fewshot_gem":  RESULTS / "dev_fewshot_or_gemini.jsonl",
    "summ_ds":      RESULTS / "dev_summarised_or_deepseek.jsonl",
    "summ_gem":     RESULTS / "dev_summarised_or_gemini.jsonl",
    "xverify_ds2gem": RESULTS / "dev_xverify_ds_to_gem.jsonl",
    "xverify_gem2ds": RESULTS / "dev_xverify_gem_to_ds.jsonl",
    "fs_summ_gem":  RESULTS / "dev_fewshot_summarised_or_gemini.jsonl",
    # fs_summ_ds сломан (URLError mid-run, 6/282 строк) — опускаем.
    "test_baseline_ds": RESULTS / "test_or_deepseek.jsonl",
}

DATA_BY_LABEL: dict[str, list[dict]] = {}
for label, path in RUNS.items():
    rows = load_run(path)
    if label.startswith("test_"):
        rows = join_gold(rows, TEST_PATH)
    else:
        rows = join_gold(rows, DEV_PATH)
    DATA_BY_LABEL[label] = rows
    print(f"{label:25s} {path.name:55s} {len(rows):4d} rows")

## 2. Распределение dev-корпуса (для текста раздела 2.5)

In [ ]:
dev_rows = [json.loads(line) for line in DEV_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
test_rows = [json.loads(line) for line in TEST_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]

def split_counts(rows: list[dict], field: str) -> dict[str, int]:
    c: dict[str, int] = defaultdict(int)
    for r in rows:
        c[r.get(field) or "—"] += 1
    return dict(sorted(c.items(), key=lambda kv: -kv[1]))

print("\n=== dev_dataset distribution ===")
print("environment:", split_counts(dev_rows, "environment"))
print("error_type :", split_counts(dev_rows, "error_type"))
print("source     :", split_counts(dev_rows, "source"))

## Таблица 4.1 — Baseline (DS vs Gem на dev и test)

In [ ]:
def stage3_row(label: str, agg: dict) -> list[str]:
    s3 = agg.get("stage3", {})
    return [
        label,
        str(agg.get("n_examples", "—")),
        str(s3.get("tp", "—")),
        str(s3.get("fp", "—")),
        str(s3.get("fn", "—")),
        fmt(s3.get("precision")),
        fmt(s3.get("recall")),
        fmt(s3.get("f1")),
        fmt(s3.get("field_accuracy")),
        fmt_pct(s3.get("hallucination_rate")),
    ]

def stage5_row(label: str, agg: dict) -> list[str]:
    s5 = agg.get("stage5", {})
    return [
        label,
        str(agg.get("n_examples", "—")),
        str(s5.get("tpv", "—")),
        str(s5.get("fpv", "—")),
        str(s5.get("fnv", "—")),
        fmt(s5.get("inconsistency_detection_rate")),
        fmt(s5.get("false_alarm_rate")),
        fmt(s5.get("combined_idr")),
        fmt(s5.get("combined_far")),
    ]

agg_baseline = {
    "DeepSeek-V3 (dev)":   agg_stages(DATA_BY_LABEL["baseline_ds"]),
    "Gemini 2.5 Flash (dev)": agg_stages(DATA_BY_LABEL["baseline_gem"]),
    "DeepSeek-V3 (test)":  agg_stages(DATA_BY_LABEL["test_baseline_ds"]),
}

stage3_rows = [stage3_row(k, v) for k, v in agg_baseline.items()]
stage5_rows = [stage5_row(k, v) for k, v in agg_baseline.items()]

table_4_1_stage3 = tex_table(
    label="tab:exp_baseline_stage3",
    caption=("Baseline: метрики извлечения тестов на этапе~3. "
             "Hallucination Rate указан в~процентах."),
    headers=["Прогон", "n", "TP", "FP", "FN", "Precision", "Recall",
             "$F_1$", "Field Acc", "Hall., \\%"],
    rows=stage3_rows,
)
table_4_1_stage5 = tex_table(
    label="tab:exp_baseline_stage5",
    caption=("Baseline: метрики самосогласованности на этапе~5. "
             "IDR\\,/\\,FAR — численная проверка; combined IDR\\,/\\,FAR "
             "— объединённая (численная~$\\lor$~семантическая)."),
    headers=["Прогон", "n", "TP$_v$", "FP$_v$", "FN$_v$",
             "IDR (p)", "FAR (p)", "comb.~IDR", "comb.~FAR"],
    rows=stage5_rows,
)
write_tex(TABLES_OUT / "table_4_1_baseline_stage3.tex", table_4_1_stage3)
write_tex(TABLES_OUT / "table_4_1_baseline_stage5.tex", table_4_1_stage5)
print(table_4_1_stage3)
print()
print(table_4_1_stage5)

### Baseline по environment (для гипотез H1, H2)

In [ ]:
def env_table(rows: list[dict], label: str) -> list[list[str]]:
    by_env = agg_by(rows, "environment")
    out = []
    for env in ["apa", "non_apa", "text", "table", "two_apa", "two_text", "no_test", "—"]:
        if env not in by_env:
            continue
        a = by_env[env]
        s3 = a.get("stage3", {})
        out.append([
            env,
            str(a["n_examples"]),
            fmt(s3.get("precision")),
            fmt(s3.get("recall")),
            fmt(s3.get("f1")),
            fmt_pct(s3.get("hallucination_rate")),
        ])
    return out

env_rows_ds = env_table(DATA_BY_LABEL["baseline_ds"], "DS")
env_rows_gem = env_table(DATA_BY_LABEL["baseline_gem"], "Gem")

# Объединяем DS и Gem в одну таблицу
env_keys = sorted({r[0] for r in env_rows_ds} | {r[0] for r in env_rows_gem})
ds_idx = {r[0]: r for r in env_rows_ds}
gem_idx = {r[0]: r for r in env_rows_gem}

combined_env_rows = []
for env in ["apa", "non_apa", "text", "table", "two_apa", "two_text", "no_test"]:
    if env not in ds_idx and env not in gem_idx:
        continue
    ds = ds_idx.get(env, ["", "—", "—", "—", "—", "—"])
    gem = gem_idx.get(env, ["", "—", "—", "—", "—", "—"])
    combined_env_rows.append([
        env,
        ds[1],
        ds[3], ds[4], ds[5],   # DS: Recall, F1, Hall
        gem[3], gem[4], gem[5],
    ])

table_4_1_env = tex_table(
    label="tab:exp_baseline_env",
    caption=("Baseline: метрики этапа~3 в~разрезе по~\\texttt{environment}. "
             "Колонки <<Recall>>, <<$F_1$>>, <<Hall., \\%>> приведены отдельно "
             "для~DeepSeek-V3 и~Gemini~2.5~Flash."),
    headers=["env", "n", "Rec.\\,DS", "$F_1$\\,DS", "Hall.\\,DS",
             "Rec.\\,Gem", "$F_1$\\,Gem", "Hall.\\,Gem"],
    rows=combined_env_rows,
)
write_tex(TABLES_OUT / "table_4_1_baseline_env.tex", table_4_1_env)
print(table_4_1_env)

## Таблица 4.2 — Эксперимент 1 (cleanup)

In [ ]:
from pipeline.stats_extractor import is_plausible, dedupe_predictions, clean_predictions
from pipeline import eval as eval_mod


def cleanup_ablation(rows: list[dict]) -> dict[str, dict]:
    """Применяет 4 варианта фильтра к predicted_tests и пересчитывает stage3."""
    variants = {
        "raw":               lambda preds: list(preds),
        "+ plausibility":    lambda preds: [t for t in preds if is_plausible(t)],
        "+ dedupe":          dedupe_predictions,
        "+ clean (both)":    clean_predictions,
    }
    out: dict[str, dict] = {}
    for vname, fn in variants.items():
        per_example = []
        n_pred = 0
        for r in rows:
            preds = r.get("predicted_tests", []) or []
            new_preds = fn(preds)
            n_pred += len(new_preds)
            per_example.append(eval_mod.eval_stage3(r.get("tests") or [], new_preds))
        agg = eval_mod.aggregate_stage3(per_example)
        agg["n_predictions"] = n_pred
        out[vname] = agg
    return out


cleanup_rows = []
for model_label, run_label in [("DeepSeek-V3", "baseline_ds"), ("Gemini 2.5 Flash", "baseline_gem")]:
    abl = cleanup_ablation(DATA_BY_LABEL[run_label])
    for vname, agg in abl.items():
        cleanup_rows.append([
            f"{model_label}, {vname}",
            str(agg.get("n_predictions", "—")),
            str(agg["tp"]), str(agg["fp"]), str(agg["fn"]),
            fmt(agg.get("precision")),
            fmt(agg.get("recall")),
            fmt(agg.get("f1")),
            fmt_pct(agg.get("hallucination_rate")),
        ])

table_4_2 = tex_table(
    label="tab:exp_cleanup",
    caption=("Эксперимент~1: вклад фильтрации неправдоподобных значений и~"
             "дедупликации в~метрики этапа~3. Все варианты применяются к~"
             "одинаковому baseline-выводу LLM без~повторных API-вызовов."),
    headers=["Модель, фильтр", "n preds", "TP", "FP", "FN",
             "Precision", "Recall", "$F_1$", "Hall., \\%"],
    rows=cleanup_rows,
)
write_tex(TABLES_OUT / "table_4_2_cleanup.tex", table_4_2)
print(table_4_2)

## Таблица 4.3 — Эксперимент 2 (CoT на этапе 4)

Headline-метрики: `primary_direction_accuracy` (stage 4) и `combined_idr`
(stage 5). Целевой класс — `error_type=wrong_conclusion_clean`.

In [ ]:
def cot_summary(label: str, rows: list[dict]) -> list[str]:
    a = agg_stages(rows)
    s4 = a.get("stage4", {})
    s5 = a.get("stage5", {})
    # Срез по wrong_conclusion_clean
    cc = [r for r in rows if r.get("error_type") == "wrong_conclusion_clean"]
    a_cc = agg_stages(cc) if cc else None
    s5_cc = (a_cc.get("stage5", {}) if a_cc else {})
    return [
        label,
        fmt(s4.get("primary_direction_accuracy")),
        fmt(s5.get("inconsistency_detection_rate")),
        fmt(s5.get("combined_idr")),
        fmt(s5_cc.get("combined_idr")) if a_cc else "—",
    ]

cot_rows = [
    cot_summary("DS, baseline",  DATA_BY_LABEL["baseline_ds"]),
    cot_summary("DS, + CoT",     DATA_BY_LABEL["cot_ds"]),
    cot_summary("Gem, baseline", DATA_BY_LABEL["baseline_gem"]),
    cot_summary("Gem, + CoT",    DATA_BY_LABEL["cot_gem"]),
]

table_4_3 = tex_table(
    label="tab:exp_cot",
    caption=("Эксперимент~2: вклад Chain-of-Thought на~этапе~4. "
             "<<Primary Acc>> — \\texttt{primary\\_direction\\_accuracy}; "
             "IDR\\,(p) — численная проверка; Combined~IDR — "
             "численная~$\\lor$~семантическая. Колонка "
             "<<Combined~IDR (clean-C)>> — на~30~синтетических примерах "
             "класса \\texttt{wrong\\_conclusion\\_clean}."),
    headers=["Прогон", "Primary Acc", "IDR (p)", "Combined IDR",
             "Combined IDR (clean-C)"],
    rows=cot_rows,
)
write_tex(TABLES_OUT / "table_4_3_cot.tex", table_4_3)
print(table_4_3)

# IDR по error_type — для гипотезы H3
def idr_by_error(rows: list[dict]) -> dict[str, dict]:
    out = {}
    for et, rrs in defaultdict_groupby(rows, "error_type").items():
        a = agg_stages(rrs)
        out[et or "—"] = {
            "n": len(rrs),
            "idr_p": a["stage5"].get("inconsistency_detection_rate"),
            "idr_combined": a["stage5"].get("combined_idr"),
        }
    return out


def defaultdict_groupby(rows: list[dict], field: str) -> dict:
    g = defaultdict(list)
    for r in rows:
        g[r.get(field)].append(r)
    return g


idr_breakdown = []
for label, run_label in [("DS, baseline", "baseline_ds"),
                          ("DS, + CoT", "cot_ds"),
                          ("Gem, baseline", "baseline_gem"),
                          ("Gem, + CoT", "cot_gem")]:
    bd = idr_by_error(DATA_BY_LABEL[run_label])
    for et in ["wrong_conclusion_clean", "wrong_conclusion", "wrong_pvalue",
               "rounding", "transcription"]:
        if et not in bd:
            continue
        v = bd[et]
        idr_breakdown.append([
            label, et, str(v["n"]),
            fmt(v["idr_p"]), fmt(v["idr_combined"]),
        ])

table_4_3_idr = tex_table(
    label="tab:exp_cot_by_error",
    caption=("Эксперимент~2: разрез Combined~IDR по~классам ошибок. "
             "Целевая ячейка~--- \\texttt{wrong\\_conclusion\\_clean}, "
             "на~которой числовая проверка бессильна по~определению."),
    headers=["Прогон", "error\\_type", "n", "IDR (p)", "Combined IDR"],
    rows=idr_breakdown,
)
write_tex(TABLES_OUT / "table_4_3_cot_by_error.tex", table_4_3_idr)
print(table_4_3_idr)

## Таблица 4.4 — Эксперимент 3 (Cross-verify)

In [ ]:
def xverify_diagnostics(rows: list[dict]) -> dict[str, int]:
    keys = ["n_proposed", "n_verified", "n_corrected", "n_removed",
            "n_added", "n_unmentioned", "n_final"]
    sums = {k: 0 for k in keys}
    for r in rows:
        diag = r.get("cross_verify_diagnostics") or {}
        for k in keys:
            sums[k] += diag.get(k, 0) or 0
    return sums


xv_rows = []
for label, run_label in [("DS, baseline",   "baseline_ds"),
                          ("Gem, baseline",  "baseline_gem"),
                          ("DS $\\to$ Gem",  "xverify_ds2gem"),
                          ("Gem $\\to$ DS",  "xverify_gem2ds")]:
    a = agg_stages(DATA_BY_LABEL[run_label])
    s3 = a["stage3"]
    s5 = a["stage5"]
    xv_rows.append([
        label,
        str(a["n_examples"]),
        str(s3.get("tp", "—")),
        str(s3.get("fp", "—")),
        str(s3.get("fn", "—")),
        fmt(s3.get("precision")),
        fmt(s3.get("recall")),
        fmt(s3.get("f1")),
        fmt(s5.get("combined_idr")),
    ])

table_4_4 = tex_table(
    label="tab:exp_xverify",
    caption=("Эксперимент~3: перекрёстная проверка LLM-as-judge. "
             "Сравниваются baseline-прогоны каждой модели и~однонаправленные "
             "верификации в~обе стороны. Combined~IDR с~этапа~5."),
    headers=["Прогон", "n", "TP", "FP", "FN", "Precision", "Recall",
             "$F_1$", "Comb.~IDR"],
    rows=xv_rows,
)
write_tex(TABLES_OUT / "table_4_4_xverify.tex", table_4_4)
print(table_4_4)

# Diagnostics — счётчики работы verifier'а
diag_rows = []
for label, run_label in [("DS $\\to$ Gem", "xverify_ds2gem"),
                          ("Gem $\\to$ DS", "xverify_gem2ds")]:
    d = xverify_diagnostics(DATA_BY_LABEL[run_label])
    diag_rows.append([
        label,
        str(d["n_proposed"]),
        str(d["n_verified"]),
        str(d["n_corrected"]),
        str(d["n_removed"]),
        str(d["n_added"]),
        str(d["n_final"]),
    ])

if any(int(r[1]) > 0 for r in diag_rows):
    table_4_4_diag = tex_table(
        label="tab:exp_xverify_diag",
        caption=("Диагностика cross-verify: сводные счётчики действий verifier'а "
                 "по~всем 282~примерам dev-сплита."),
        headers=["Прогон", "n\\_proposed", "n\\_verified", "n\\_corrected",
                 "n\\_removed", "n\\_added", "n\\_final"],
        rows=diag_rows,
    )
    write_tex(TABLES_OUT / "table_4_4_xverify_diag.tex", table_4_4_diag)
    print(table_4_4_diag)
else:
    print("(cross_verify_diagnostics пусто — диагностическая таблица опущена)")

## Таблица 4.5 — Эксперимент 4 (Summarisation)

In [ ]:
def summ_stats(rows: list[dict]) -> dict[str, float | None]:
    """Среднее ratio и сумма n_calls по примерам, где сжатие сработало."""
    n_used = 0
    n_calls_total = 0
    ratios = []
    for r in rows:
        s = r.get("summarization_stats") or {}
        if s.get("n_calls", 0) > 0:
            n_used += 1
            n_calls_total += s["n_calls"]
            if s.get("ratio") is not None:
                ratios.append(s["ratio"])
    return {
        "n_used": n_used,
        "n_calls_total": n_calls_total,
        "ratio_mean": (sum(ratios) / len(ratios)) if ratios else None,
    }


def summ_split(rows_baseline: list[dict], rows_summ: list[dict], source_filter: str | None = None):
    if source_filter:
        rows_baseline = [r for r in rows_baseline if r.get("source") == source_filter]
        rows_summ = [r for r in rows_summ if r.get("source") == source_filter]
    return agg_stages(rows_baseline), agg_stages(rows_summ), summ_stats(rows_summ)


summ_rows = []
for model_label, base_run, summ_run in [
    ("DeepSeek-V3", "baseline_ds", "summ_ds"),
    ("Gemini 2.5 Flash", "baseline_gem", "summ_gem"),
]:
    for src_label, src_filter in [("synthetic", "synthetic"), ("real", "real"), ("all", None)]:
        a_base, a_summ, ss = summ_split(
            DATA_BY_LABEL[base_run], DATA_BY_LABEL[summ_run], src_filter
        )
        s3_b = a_base["stage3"]; s3_s = a_summ["stage3"]
        summ_rows.append([
            f"{model_label}, {src_label}",
            str(a_base["n_examples"]),
            fmt(s3_b.get("recall")),       fmt(s3_s.get("recall")),
            fmt(s3_b.get("f1")),           fmt(s3_s.get("f1")),
            fmt_pct(s3_b.get("hallucination_rate")),
            fmt_pct(s3_s.get("hallucination_rate")),
            fmt(ss["ratio_mean"]),
            str(ss["n_used"]),
        ])

table_4_5 = tex_table(
    label="tab:exp_summarisation",
    caption=("Эксперимент~4: вклад суммаризации секций на~этапе~2. "
             "Сравнение baseline (без~суммаризации) и~режима \\texttt{--summarise} "
             "по~трём срезам: только синтетика, только real-статьи, всё. "
             "<<ratio>>~--- средний коэффициент сжатия секций; "
             "<<n used>>~--- число примеров, на~которых хотя~бы одна секция "
             "оказалась длиннее~500~символов и~была сжата."),
    headers=["Прогон / src", "n", "Recall (b)", "Recall (s)",
             "$F_1$ (b)", "$F_1$ (s)",
             "Hall., \\% (b)", "Hall., \\% (s)",
             "ratio", "n used"],
    rows=summ_rows,
)
write_tex(TABLES_OUT / "table_4_5_summarisation.tex", table_4_5)
print(table_4_5)

## Таблица 4.6 — Эксперимент 5 (Few-shot для таблиц)

In [ ]:
def fewshot_split(rows: list[dict], env_filter: str | None = None) -> dict:
    if env_filter:
        rows = [r for r in rows if r.get("environment") == env_filter]
    a = agg_stages(rows)
    fp = fp_category_counts(rows)
    return {**a, "fp_categories": fp}


fewshot_rows = []
for model_label, base_run, fs_run in [
    ("DeepSeek-V3", "baseline_ds", "fewshot_ds"),
    ("Gemini 2.5 Flash", "baseline_gem", "fewshot_gem"),
]:
    for env_label, env_filter in [("all", None), ("table", "table"), ("real", None)]:
        if env_label == "real":
            base_rows = [r for r in DATA_BY_LABEL[base_run] if r.get("source") == "real"]
            fs_rows = [r for r in DATA_BY_LABEL[fs_run] if r.get("source") == "real"]
        else:
            base_rows = DATA_BY_LABEL[base_run]
            fs_rows = DATA_BY_LABEL[fs_run]
            if env_filter:
                base_rows = [r for r in base_rows if r.get("environment") == env_filter]
                fs_rows = [r for r in fs_rows if r.get("environment") == env_filter]
        a_b = agg_stages(base_rows)
        a_f = agg_stages(fs_rows)
        s3_b = a_b["stage3"]; s3_f = a_f["stage3"]
        fp_b = fp_category_counts(base_rows)
        fp_f = fp_category_counts(fs_rows)
        fewshot_rows.append([
            f"{model_label}, {env_label}",
            str(a_b["n_examples"]),
            fmt(s3_b.get("recall")), fmt(s3_f.get("recall")),
            fmt(s3_b.get("f1")),     fmt(s3_f.get("f1")),
            str(fp_b.get("duplicate", 0)),
            str(fp_f.get("duplicate", 0)),
        ])

table_4_6 = tex_table(
    label="tab:exp_fewshot",
    caption=("Эксперимент~5: вклад few-shot инжекта для~таблиц на~этапе~3. "
             "Сравнение baseline и~few-shot режимов на~трёх срезах "
             "(all dev / только \\texttt{environment=table} / только real-статьи). "
             "Колонка <<duplicate (b/f)>>~--- число FP-категории "
             "\\texttt{duplicate} в~baseline и~few-shot прогонах "
             "(см.~\\texttt{classify\\_fp} в~\\texttt{pipeline/eval.py})."),
    headers=["Прогон / срез", "n", "Recall (b)", "Recall (f)",
             "$F_1$ (b)", "$F_1$ (f)",
             "duplicate (b)", "duplicate (f)"],
    rows=fewshot_rows,
)
write_tex(TABLES_OUT / "table_4_6_fewshot.tex", table_4_6)
print(table_4_6)

## Таблица 4.7 — Сравнение моделей

In [ ]:
def compact_summary(label: str, rows: list[dict]) -> list[str]:
    a = agg_stages(rows)
    s3, s4, s5 = a["stage3"], a["stage4"], a["stage5"]
    return [
        label,
        str(a["n_examples"]),
        fmt(s3.get("f1")),
        fmt(s4.get("primary_direction_accuracy")),
        fmt(s5.get("combined_idr")),
        fmt_pct(s3.get("hallucination_rate")),
    ]


model_compare_rows = [
    compact_summary("DS, baseline",      DATA_BY_LABEL["baseline_ds"]),
    compact_summary("DS, + CoT",         DATA_BY_LABEL["cot_ds"]),
    compact_summary("DS, + few-shot",    DATA_BY_LABEL["fewshot_ds"]),
    compact_summary("DS, + summarised",  DATA_BY_LABEL["summ_ds"]),
    compact_summary("DS $\\to$ Gem",     DATA_BY_LABEL["xverify_ds2gem"]),
    compact_summary("Gem, baseline",     DATA_BY_LABEL["baseline_gem"]),
    compact_summary("Gem, + CoT",        DATA_BY_LABEL["cot_gem"]),
    compact_summary("Gem, + few-shot",   DATA_BY_LABEL["fewshot_gem"]),
    compact_summary("Gem, + summarised", DATA_BY_LABEL["summ_gem"]),
    compact_summary("Gem $\\to$ DS",     DATA_BY_LABEL["xverify_gem2ds"]),
]

table_4_7 = tex_table(
    label="tab:exp_models",
    caption=("Сводное сравнение DeepSeek-V3 и~Gemini~2.5~Flash по~всем "
             "стратегиям конфигурации на~dev-сплите. Headline-метрики: "
             "$F_1$ извлечения (этап~3), Primary Direction Accuracy "
             "(этап~4), Combined~IDR (этап~5), уровень галлюцинаций "
             "($\\downarrow$~лучше)."),
    headers=["Прогон", "n", "$F_1$", "Primary Acc",
             "Comb.~IDR", "Hall., \\%"],
    rows=model_compare_rows,
)
write_tex(TABLES_OUT / "table_4_7_models.tex", table_4_7)
print(table_4_7)

## Таблицы 4.9 и 4.11 — Анализ ошибок (FP-категории и IDR по error_type)

In [ ]:
def fp_dist_row(label: str, rows: list[dict]) -> list[str]:
    fp = fp_category_counts(rows)
    total = sum(fp.values())
    keys = ["duplicate", "wrong_test_type", "off_stat_value", "complete_fabrication"]
    return [label, str(total)] + [
        f"{fp.get(k, 0)} ({fmt_pct(fp.get(k, 0) / total) if total else '—'}\\%)"
        for k in keys
    ]


fp_rows = [
    fp_dist_row("DS, baseline",  DATA_BY_LABEL["baseline_ds"]),
    fp_dist_row("Gem, baseline", DATA_BY_LABEL["baseline_gem"]),
    fp_dist_row("DS, + CoT",     DATA_BY_LABEL["cot_ds"]),
    fp_dist_row("Gem, + CoT",    DATA_BY_LABEL["cot_gem"]),
    fp_dist_row("DS, + few-shot", DATA_BY_LABEL["fewshot_ds"]),
    fp_dist_row("Gem, + few-shot", DATA_BY_LABEL["fewshot_gem"]),
]

table_4_9 = tex_table(
    label="tab:exp_fp_categories",
    caption=("Распределение ложных срабатываний (FP) по~категориям "
             "\\texttt{classify\\_fp} (см.~\\texttt{pipeline/eval.py}): "
             "\\texttt{duplicate}, \\texttt{wrong\\_test\\_type}, "
             "\\texttt{off\\_stat\\_value}, \\texttt{complete\\_fabrication}. "
             "Числа~--- абсолютное количество и~доля от~всех FP."),
    headers=["Прогон", "FP всего", "duplicate", "wrong\\_type",
             "off\\_value", "fabrication"],
    rows=fp_rows,
)
write_tex(TABLES_OUT / "table_4_9_fp_categories.tex", table_4_9)
print(table_4_9)

# IDR / FAR по error_type — таблица 4.11
def idr_far_by_error_row(label: str, rows: list[dict]) -> list[list[str]]:
    out = []
    by_err = defaultdict_groupby(rows, "error_type")
    for et in ["wrong_conclusion_clean", "wrong_conclusion", "wrong_pvalue",
               "rounding", "transcription"]:
        rrs = by_err.get(et, [])
        if not rrs:
            continue
        a = agg_stages(rrs)
        s5 = a["stage5"]
        out.append([
            label, et, str(len(rrs)),
            fmt(s5.get("inconsistency_detection_rate")),
            fmt(s5.get("combined_idr")),
            fmt(s5.get("false_alarm_rate")),
            fmt(s5.get("combined_far")),
        ])
    return out


idr_far_rows = []
for label, run_label in [("DS, baseline", "baseline_ds"),
                          ("DS, + CoT",   "cot_ds"),
                          ("Gem, baseline", "baseline_gem"),
                          ("Gem, + CoT",  "cot_gem")]:
    idr_far_rows.extend(idr_far_by_error_row(label, DATA_BY_LABEL[run_label]))

table_4_11 = tex_table(
    label="tab:exp_idr_by_error",
    caption=("Inconsistency Detection Rate и~False Alarm Rate в~разрезе "
             "по~внесённому \\texttt{error\\_type}. Headline-сравнение"
             "~--- ячейка \\texttt{wrong\\_conclusion\\_clean}~$\\times$~CoT, "
             "где~численная проверка бессильна по~определению."),
    headers=["Прогон", "error\\_type", "n",
             "IDR (p)", "Comb.~IDR", "FAR (p)", "Comb.~FAR"],
    rows=idr_far_rows,
)
write_tex(TABLES_OUT / "table_4_11_idr_by_error.tex", table_4_11)
print(table_4_11)

# F1 по environment — таблица 4.10
def f1_env_rows() -> list[list[str]]:
    out = []
    envs = ["apa", "non_apa", "text", "table", "two_apa", "two_text", "no_test"]
    for env in envs:
        ds_a = agg_stages([r for r in DATA_BY_LABEL["baseline_ds"] if r.get("environment") == env])
        gem_a = agg_stages([r for r in DATA_BY_LABEL["baseline_gem"] if r.get("environment") == env])
        if ds_a["n_examples"] == 0 and gem_a["n_examples"] == 0:
            continue
        out.append([
            env, str(ds_a["n_examples"]),
            fmt(ds_a["stage3"].get("f1")),
            fmt(ds_a["stage3"].get("recall")),
            fmt_pct(ds_a["stage3"].get("hallucination_rate")),
            fmt(gem_a["stage3"].get("f1")),
            fmt(gem_a["stage3"].get("recall")),
            fmt_pct(gem_a["stage3"].get("hallucination_rate")),
        ])
    return out


table_4_10 = tex_table(
    label="tab:exp_env_breakdown",
    caption=("Метрики этапа~3 в~разрезе по~\\texttt{environment} для~baseline. "
             "Колонки <<DS>> и~<<Gem>>~--- DeepSeek-V3 и~Gemini~2.5~Flash."),
    headers=["env", "n", "$F_1$\\,DS", "Rec.\\,DS", "Hall.\\,DS",
             "$F_1$\\,Gem", "Rec.\\,Gem", "Hall.\\,Gem"],
    rows=f1_env_rows(),
)
write_tex(TABLES_OUT / "table_4_10_env_breakdown.tex", table_4_10)
print(table_4_10)

## Сводный markdown-отчёт для ревью

In [ ]:
def summary_md() -> str:
    out = ["# Глава 4 — сводка результатов\n"]
    out.append("Сгенерировано `notebooks/chapter4_analysis.py`.\n")
    out.append("## Baseline (dev)\n")
    for label, run in [("DeepSeek-V3", "baseline_ds"), ("Gemini 2.5 Flash", "baseline_gem")]:
        a = agg_stages(DATA_BY_LABEL[run])
        s3 = a["stage3"]; s5 = a["stage5"]
        out.append(
            f"- **{label}**: n={a['n_examples']}, "
            f"Precision={fmt(s3.get('precision'))}, "
            f"Recall={fmt(s3.get('recall'))}, "
            f"F1={fmt(s3.get('f1'))}, "
            f"Hallucination={fmt_pct(s3.get('hallucination_rate'))}%, "
            f"IDR(p)={fmt(s5.get('inconsistency_detection_rate'))}, "
            f"Combined IDR={fmt(s5.get('combined_idr'))}"
        )
    out.append("\n## CoT — clean-C класс\n")
    for label, run in [("DS, + CoT", "cot_ds"), ("Gem, + CoT", "cot_gem")]:
        cc = [r for r in DATA_BY_LABEL[run] if r.get("error_type") == "wrong_conclusion_clean"]
        a = agg_stages(cc) if cc else None
        if a:
            out.append(f"- **{label}** clean-C: n={len(cc)}, Combined IDR={fmt(a['stage5'].get('combined_idr'))}")
    out.append("\n## Cross-verify\n")
    for label, run in [("DS $\\to$ Gem", "xverify_ds2gem"), ("Gem $\\to$ DS", "xverify_gem2ds")]:
        a = agg_stages(DATA_BY_LABEL[run])
        d = xverify_diagnostics(DATA_BY_LABEL[run])
        out.append(
            f"- **{label}**: F1={fmt(a['stage3'].get('f1'))}, "
            f"removed={d['n_removed']}, added={d['n_added']}, "
            f"corrected={d['n_corrected']}"
        )
    out.append("\n## Файлы таблиц\n")
    for tex in sorted(TABLES_OUT.glob("table_4_*.tex")):
        out.append(f"- `{tex.relative_to(REPO)}`")
    return "\n".join(out) + "\n"


md_path = TABLES_OUT / "chapter4_summary.md"
md_path.write_text(summary_md(), encoding="utf-8")
print(f"\nSummary written to {md_path}")
print((md_path).read_text())